# JEOL elemental maps (.pts)

Import required commands and define path were demo data is stored. Adapt this for your installation.

In [ ]:
from pathlib import PurePath
import numpy as np
import matplotlib.pyplot as plt
from JEOL_eds import JEOL_pts
import JEOL_eds.utils as JU

base = PurePath("/home/alxneit/private/data/46_26/13A9/3/Sample/00_View000")

Load example data (`View000_0000007.pts`). This data set contains 80 frames with size 512x512 and 4000 energy channels. With the default data type (`uint16`) this results in about 168 GB! On top of this, there are also 80 images stored (one per frame) that will be used to compensate from sample movement. To minimize memory consumption, we first load only the meta data that contain the reference spectrum (sum over all frames and positions) and plot it to locate the maximum energy that contains relevant data.

In [ ]:
dc = JEOL_pts(base / "View000_0000007.pts", only_metadata=True)
JU.plot_spectrum(dc.ref_spectrum, log_y=True)

We use an energy cutoff of 12keV. This reduces the memory required by 2/3. To further reduce the memory consumption we rebin (by 4x4) the data while loading. Together, this reduces the size of the data cube to about 3 GB.

Load the data (individual frames) including the drift images. The data cube will contain 80 frames each 128x128 pixels and 1200 energy channels.

In [ ]:
dc = JEOL_pts(base / "View000_0000007.pts", read_drift="yes", split_frames=True, rebin=(4, 4), E_cutoff=12.0)
dc.dcube.shape

Before we continue, check, if drift corrections are required. We have two options: First, make a mp4 movie with the drift image in the left pane and the total edx intensity in the right pane. Use the file name of the data but with `.pts` replaced by `mp4`. Check out the movie to realize that the sample jumps around a bit. In the output, only the last frame is shown.

In [ ]:
dc.make_movie()

Second, we calculate the image shift for each frame (from cross correlation with a reference) using the drift images (`align_src="drift_images"`) or data (`align_src="data"`) to which we might apply a Wiener filter (`filtered=True`). We use the 1st frame (`reference=0`) as reference. Ignore any warnings.

In [ ]:
dc.shifts(align_src="drift_images", filtered=True, reference=9)

We could also visualize the shifts as 2D probability distribution (histogram).

In [ ]:
m, e = dc.drift_statistics(align_src="drift_images", filtered=True, reference=9)
plt.imshow(m, extent=e)

Apply the drift correction. **Careful**, the overwrites the data cube (`JEOL_pts.dcube`) and the drift images (`JEOL_pts.drift_images`). Make a movie again (this time, specify a file name) to observe the result.

In [ ]:
dc.correct_drift(align_src="drift_images", apply_filter=True, reference=9)
dc.make_movie(base / "corrected.mp4")

Carbon writing seems to have begun a some point during data acquisition. Follow the intensity of the carbon Ka line between 0.225 keV and 0.325 keV (manually determined from a zoomed-in version of the reference spectrum).

In [ ]:
I_carbon = dc.time_series(interval=(0.225, 0.325), energy=True)
JU.plot_tseries(I_carbon, M_ticks=(9, None))

Extract an elemental map for Pt using the Ma1 line at 2.05 keV. The line might be partially overlapped by the Au Ma1 line at 2.12 keV (intrumental background). The Pt L lines are visible but very weak. 

In [ ]:
Pt = dc.map(interval=(2.0, 2.2), energy=True)
JU.plot_map(Pt, "purple", label="Pt")

Try to improve the map by adding the contributions of the different Pt La lines. Note, that this also adds substantial amounts of noise.

In [ ]:
Pt += dc.map(interval=(9.3, 9.6), energy=True)
JU.plot_map(Pt, "purple", label="Pt")

Overlay the map onto one of the drift images to help with the interpretation.

In [ ]:
JU.create_overlay([Pt], ["purple"], BG_image=dc.drift_images[9])

Extract the spectra of two regions, one containing a single Pt nanoparticle (`circle`) and one containing the empty support (`square`). Show the regions for verification and then plot the corresponding spectrum. Only use frames 0 to 55 where no carbon writing occured.

In [ ]:
circle = (43, 44, 8)
JU.show_ROI(dc.drift_images[9], circle)

In [ ]:
JU.plot_spectrum(dc.spectrum(ROI=circle, frames=list(range(55))))

In [ ]:
square = (80, 120, 35, 75)
JU.show_ROI(dc.drift_images[9], square)

In [ ]:
JU.plot_spectrum(dc.spectrum(ROI=square, frames=list(range(55))))

Finally, extract some profiles along an arbitrary line (on an image or map). First, verify line definition then extract Pt and and Al profiles and plot them. For this we reuse the platinum elemental map and create an aluminum map using Al Ka at 1.5 keV.

In [ ]:
line = (20, 34, 120, 73)
JU.show_line(dc.drift_images[9], line, linewidth=10)

In [ ]:
pPt = JU.get_profile(Pt, line, linewidth=10, reduce_func=np.sum)

Al = dc.map(interval=(1.45, 1.55), energy=True)
pAl = JU.get_profile(Al, line, linewidth=10, reduce_func=np.sum)

x = range(pAl.shape[0])

plt.plot(x, pAl, x, pPt)